# 11.2 Дизайн теста — код страницы

Аналитика 101 · блок «Дата аналитика» · тема 11 «A/B-тестирование» · страница 11.2.

В ноутбуке три блока кода страницы: параметры из данных, MDE из денег и размер выборки вручную; то же в statsmodels с мощностью по факту; проверка мощности симуляцией. Четвёртый блок страницы — шаблон документа дизайна — лежит рядом файлом `test_design_template.md`. Данные учебные: показы и сессии экспериментов из общего набора темы `data/ab/`.

**Как работать.** Выполняйте ячейки по порядку сверху вниз: каждый блок продолжает предыдущий.

Вывод получен на Python 3.13, pandas 3.0, NumPy 2.5, SciPy 1.18 и statsmodels 0.15.

## Где лежат данные

Код страницы читает файлы по пути от корня папки с материалами: `data/ab/exposures.csv` и `data/ab/sessions.parquet`. Ноутбук лежит в папке страницы, поэтому первая ячейка поднимается до папки, в которой есть `data/`, и делает её рабочей.

In [1]:
import os
import pathlib
import platform

import numpy as np
import pandas as pd
import scipy
import statsmodels

root = pathlib.Path.cwd()
while not (root / "data" / "ab").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/ab/
assert (root / "data" / "ab" / "exposures.csv").exists(), \
    "Не найдена папка data/ab/. Скачайте общий набор темы 11 и положите data/ рядом с папкой страницы"
os.chdir(root)

print("Python", platform.python_version(), "| pandas", pd.__version__, "| NumPy", np.__version__,
      "| SciPy", scipy.__version__, "| statsmodels", statsmodels.__version__)

Python 3.13.18 | pandas 3.0.5 | NumPy 2.5.3 | SciPy 1.18.1 | statsmodels 0.15.0


## Код: параметры из данных, MDE из денег и выборка вручную

Трое ботов исключены списком: как их находят, показывает страница 11.3.

### Параметры из данных, MDE из денег и размер выборки вручную

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# 1. Участники тестов: группа — из журнала показов, боты исключены
bots = ["u_025389", "u_053191", "u_055948"]
exp = pd.read_csv("data/ab/exposures.csv", parse_dates=["exposed_at"])
exp = exp[~exp.user_id.isin(bots)]
ses = pd.read_parquet("data/ab/sessions.parquet")
users = (ses.groupby(["experiment", "user_id"])
            .agg(conv=("converted", "max"), revenue=("revenue", "sum"))
            .reset_index()
            .merge(exp, on=["experiment", "user_id"]))

# 2. База — 4 недели без изменения: группа A checkout_v2
a = users[(users.experiment == "checkout_v2") & (users.group == "A")]
p0, mu, sd = a.conv.mean(), a.revenue.mean(), a.revenue.std()
print(f"конверсия A: {p0:.4f}   выручка на человека: {mu:.1f} ₽, σ = {sd:.0f} ₽, σ/среднее = {sd / mu:.2f}")

# 3. MDE из денег: цена пункта конверсии против цены изменения
people = (users.experiment == "checkout_v2").sum()   # участников за 4 недели
margin = 783                         # маржинальный доход обычного заказа, ₽ (страница 10.1)
point = people * 0.01 * margin * 13  # +1 п. п. за год: 13 окон по 4 недели
cost = 5_000_000                     # разработка и поддержка за год, ₽ — условная цифра
mde = round(cost / point / 100, 4)   # в долях, до сотых пункта: 0,0086 = 0,86 п. п.
print(f"пункт конверсии: {point / 1e6:.2f} млн ₽ в год → MDE = {mde * 100:.2f} п. п. = {mde / p0:+.1%}")

# 4. Формула: n = (z₁₋α/₂ + z₁₋β)² · (σ²_A + σ²_B) / MDE² — людей в КАЖДОЙ группе
alpha, power = 0.05, 0.80
z = norm.ppf(1 - alpha / 2) + norm.ppf(power)        # 1,960 + 0,842
p1 = p0 + mde
n_conv = np.ceil(z**2 * (p0 * (1 - p0) + p1 * (1 - p1)) / mde**2)
d_rev = round(mu * mde / p0)                         # тот же относительный эффект для выручки, ₽
n_rev = np.ceil(z**2 * 2 * sd**2 / d_rev**2)
print(f"конверсия: {n_conv:.0f} в группе")
print(f"выручка (+{d_rev:.0f} ₽): {n_rev:.0f} в группе — в {n_rev / n_conv:.1f} раза больше")

# 5. Срок: поток новых участников и целые недели
day = (exp.exposed_at.dt.tz_convert("Europe/Minsk").dt.date)[exp.experiment == "checkout_v2"]
flow = day.value_counts().sort_index()
print("новых участников в день, первая неделя:", flow.iloc[:7].tolist())
print("                         последняя неделя:", flow.iloc[-7:].tolist())
for n in (n_conv, n_rev):
    days = np.ceil(2 * n / flow.mean())
    print(f"{2 * n:.0f} человек при {flow.mean():.0f} в день → {days:.0f} дней → {np.ceil(days / 7):.0f} нед.")

конверсия A: 0.1408   выручка на человека: 998.9 ₽, σ = 6912 ₽, σ/среднее = 6.92
пункт конверсии: 5.78 млн ₽ в год → MDE = 0.86 п. п. = +6.1%
конверсия: 26329 в группе
выручка (+61 ₽): 201535 в группе — в 7.7 раза больше
новых участников в день, первая неделя: [7920, 5982, 4484, 3760, 3157, 3293, 2764]
                         последняя неделя: [891, 833, 860, 834, 760, 828, 903]
52658 человек при 2028 в день → 26 дней → 4 нед.
403070 человек при 2028 в день → 199 дней → 29 нед.


Базой служит группа A самого checkout_v2: это и есть «4 недели без изменения». В жизни её берут из прошлого периода той же длины. 783 ₽ — маржинальный доход обычного заказа своего товара из учебной выгрузки темы 10, 5 млн ₽ — условная цена изменения.

## Код: то же в statsmodels

### То же в statsmodels: размер выборки, MDE по выборке, мощность по факту

In [3]:
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import proportion_effectsize

pw = NormalIndPower()
# Доля: statsmodels меряет эффект в арксинусах — это h Коэна
h = proportion_effectsize(p1, p0)
print(f"конверсия, h = {h:.4f}:  {pw.solve_power(h, alpha=0.05, power=0.8):.0f} в группе")
# Та же формула, что вручную: эффект в долях σ̄ = √((σ²_A + σ²_B) / 2)
d_conv = mde / np.sqrt((p0 * (1 - p0) + p1 * (1 - p1)) / 2)
print(f"конверсия, Δ/σ̄ = {d_conv:.4f}: {pw.solve_power(d_conv, alpha=0.05, power=0.8):.0f} в группе")
# Среднее: d = Δ / σ; t-тест вместо z почти ничего не меняет
d = d_rev / sd
print(f"выручка, d = {d:.4f}: z {pw.solve_power(d, alpha=0.05, power=0.8):.0f}, "
      f"t {TTestIndPower().solve_power(d, alpha=0.05, power=0.8):.0f} в группе")

# Обратная задача: какой эффект видит выборка, которая набралась в checkout_v2
c = users[users.experiment == "checkout_v2"]
nA, nB = (c.group == "A").sum(), (c.group == "B").sum()
es = pw.solve_power(nobs1=nA, ratio=nB / nA, alpha=0.05, power=0.8)    # в стандартных единицах
p_min = np.sin(np.arcsin(np.sqrt(p0)) + es / 2) ** 2                   # из арксинусов обратно в долю
print(f"\nвыборка {nA} + {nB}: MDE конверсии {(p_min - p0) * 100:.2f} п. п., "
      f"выручки {es * sd:.0f} ₽ ({es * sd / mu:+.1%})")

# Что получили и какая мощность была у дизайна для эффекта такого размера
b = c[c.group == "B"]
for name, eff, es_obs in [
        ("конверсия", f"{(b.conv.mean() - p0) * 100:+.2f} п. п.", proportion_effectsize(b.conv.mean(), p0)),
        ("выручка", f"{b.revenue.mean() / mu - 1:+.1%}", (b.revenue.mean() - mu) / sd)]:
    print(f"{name}: {eff}, мощность для такого эффекта {pw.power(es_obs, nobs1=nA, alpha=0.05, ratio=nB / nA):.2f}")

конверсия, h = 0.0244:  26327 в группе
конверсия, Δ/σ̄ = 0.0244: 26329 в группе
выручка, d = 0.0088: z 201534, t 201535 в группе

выборка 28439 + 28358: MDE конверсии 0.83 п. п., выручки 163 ₽ (+16.3%)
конверсия: +1.17 п. п., мощность для такого эффекта 0.98
выручка: +7.8%, мощность для такого эффекта 0.27


Ячейка продолжает предыдущую: p0, p1, mde, d_rev, sd, users уже в памяти. solve_power возвращает nobs1 — людей в первой группе; вторая — nobs1 · ratio.

## Код: проверка мощности симуляцией

Зерно генератора зафиксировано: повторный запуск даёт тот же вывод.

### Проверка мощности симуляцией: тысячи тестов с известным эффектом

In [4]:
from scipy.stats import ttest_ind

rng = np.random.default_rng(1102)
R = 2000                                    # тестов в каждой симуляции


def z_share(xa, xb, n):
    """Доля тестов с p < 0,05: z-тест для долей, xa и xb — числа покупателей в группах по n."""
    pa, pb, pp = xa / n, xb / n, (xa + xb) / (2 * n)
    zz = (pb - pa) / np.sqrt(pp * (1 - pp) * 2 / n)
    return np.mean(2 * norm.sf(np.abs(zz)) < 0.05)


# 1. Конверсия: R тестов по n_conv человек в группе
n = int(n_conv)
aa = z_share(rng.binomial(n, p0, R), rng.binomial(n, p0, R), n)          # эффекта нет
ab = z_share(rng.binomial(n, p0, R), rng.binomial(n, p0 + mde, R), n)    # эффект = MDE
print(f"конверсия, {n} в группе: A/A значимы {aa:.1%}, A/B с эффектом MDE — {ab:.1%}")


# 2. Выручка: людей берём из группы A с возвращением — хвост настоящий
rev = a.revenue.to_numpy()


def rev_share(n, lift, reps=R):
    hits = 0
    for _ in range(reps):
        x, y = rng.choice(rev, n), rng.choice(rev, n) * lift
        hits += ttest_ind(y, x, equal_var=False).pvalue < 0.05
    return hits / reps


lift = b.revenue.mean() / mu                # +7,8%, как вышло в checkout_v2
need = int(np.ceil(z**2 * (1 + lift**2) * sd**2 / (mu * (lift - 1)) ** 2))   # σ_B = lift · σ_A
print(f"выручка, {nA} в группе: A/A значимы {rev_share(nA, 1):.1%}, "
      f"A/B с эффектом {lift - 1:+.1%} — {rev_share(nA, lift):.1%}")
print(f"выручка, {need} в группе: A/B с эффектом {lift - 1:+.1%} — {rev_share(need, lift):.1%}")

конверсия, 26329 в группе: A/A значимы 4.5%, A/B с эффектом MDE — 81.2%


выручка, 28439 в группе: A/A значимы 5.9%, A/B с эффектом +7.8% — 26.6%


выручка, 133658 в группе: A/B с эффектом +7.8% — 78.4%


Ячейка продолжает две предыдущие. Погрешность доли по 2 000 тестов — около 1 п. п. Зерно генератора фиксировано, повторный запуск даёт тот же вывод.